Exact source of Kaggle kernel `tfb-swap-t1b` (pulled from Kaggle).

In [ ]:
"""Start from the exact 0.85309 file; swap Task 1 rows (T1 v5 B: + low-rank features) and optionally Task 2 rows (v10 CatBoost onset)."""
import glob, numpy as np, pandas as pd
REL = glob.glob('/kaggle/input/**/kaggle_public', recursive=True)[0]
base = pd.read_csv(glob.glob('/kaggle/input/**/tfb-swap-v5/**/submission_og_noprior.csv', recursive=True)[0])
key = pd.read_csv(f'{REL}/submission_key.csv', dtype=str)
# ---- Task 1 rows
st = pd.read_csv(glob.glob('/kaggle/input/**/tfb-t1v5-full/**/state_submission_B.csv', recursive=True)[0], dtype={'station_id': str, 'link_id': str})
ks = key[key.task == 'state'][['submission_id', 'panel', 'timestamp', 'station_id', 'link_id', 'mask_regime']].copy()
for d in (st, ks):
    d['timestamp'] = pd.to_datetime(d.timestamp, utc=True)
m = ks.merge(st, on=['panel', 'timestamp', 'station_id', 'link_id', 'mask_regime'], how='left', validate='one_to_one')
assert m.speed_kmh.notna().all() and m.flow_vph.notna().all(), 'unmatched state keys'
sv = pd.Series(m.speed_kmh.to_numpy(), index=m.submission_id.astype('int64').to_numpy())
sq = pd.Series(m.flow_vph.to_numpy(), index=m.submission_id.astype('int64').to_numpy())
iss = (base.task == 'state').to_numpy(); sid = base.loc[iss, 'submission_id'].to_numpy()
t1 = base.copy()
t1.loc[iss, 'speed_kmh'] = sv.reindex(sid).to_numpy(); t1.loc[iss, 'flow_vph'] = sq.reindex(sid).to_numpy()
low = (t1.loc[iss, 'flow_vph'] < 50).mean()
print('T1 swap: rms speed change', round(float(np.sqrt(np.mean((t1.loc[iss, 'speed_kmh'] - base.loc[iss, 'speed_kmh']) ** 2))), 3),
      'rms flow change', round(float(np.sqrt(np.mean((t1.loc[iss, 'flow_vph'] - base.loc[iss, 'flow_vph']) ** 2))), 2), 'flow<50 share', round(float(low), 5))
# ---- Task 2 rows from v10 main
q = pd.read_csv(glob.glob('/kaggle/input/**/tfb-t2v10/**/queue_submission.csv', recursive=True)[0], dtype={'window_id': str, 'link_id': str})
kq = key[key.task == 'queue'][['submission_id', 'window_id', 'timestamp', 'link_id']].copy()
for d in (q, kq):
    d['timestamp'] = pd.to_datetime(d.timestamp, utc=True)
mq = kq.merge(q, on=['window_id', 'timestamp', 'link_id'], how='left', validate='one_to_one'); assert mq.queue_pred.notna().all()
qv = pd.Series(mq.queue_pred.astype(int).to_numpy(), index=mq.submission_id.astype('int64').to_numpy())
isq = (base.task == 'queue').to_numpy()
both = t1.copy(); both.loc[isq, 'queue_pred'] = qv.reindex(base.loc[isq, 'submission_id'].to_numpy()).to_numpy()
sample = pd.read_csv(f'{REL}/sample_submission.csv', usecols=['submission_id']).submission_id.to_numpy()
for name, df in (('T1B', t1), ('T1B_T2cat', both)):
    assert len(df) == len(sample) and (df.submission_id.to_numpy() == sample).all()
    assert np.isfinite(df[['speed_kmh', 'flow_vph', 'queue_pred', 'path_flow']].to_numpy()).all()
    assert (df[['speed_kmh', 'flow_vph', 'path_flow']] >= 0).all().all() and df.loc[isq, 'queue_pred'].isin([0, 1]).all()
    df['queue_pred'] = df.queue_pred.astype(int)
    df.to_csv(f'/kaggle/working/submission_{name}.csv', index=False)
    print(name, 'OK', len(df), 'queue agreement with 0.85309', round(float((df.loc[isq, 'queue_pred'].to_numpy() == base.loc[isq, 'queue_pred'].to_numpy()).mean()), 4))
